# Building RAG Pipelines — C8-W4-S1
## Notebook 01: RAG Foundations
**Duration:** 15 min &nbsp;|&nbsp; **Mode:** Conceptual &nbsp;|&nbsp; upGrad Live Session

> We build RAG as a **modular pipeline**. Every stage is taught **WHY → WHAT → HOW**,
> and we keep asking the session's guiding question: *"What happens if this step is
> poorly designed?"* Frameworks (LangChain) appear only as a **parallel mapping** —
> they abstract the mechanics but **do not eliminate the design decisions**.

![pipeline](https://dummyimage.com/1000x70/1f2937/ffffff&text=Loading+%E2%86%92+Chunking+%E2%86%92+Retrieval+%E2%86%92+Augmentation+%E2%86%92+Generation+%E2%86%92+Evaluation)

In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first.
# ============================================================
# It (1) installs dependencies, (2) makes the `rag_pipeline` package importable,
# and (3) locates the sample corpus. Everything below runs even with NO API key,
# because the package falls back to a deterministic offline "mock" provider.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

# >>> INSTRUCTOR: set this to your repo URL so Colab can fetch the package. <<<
REPO_URL = "https://github.com/baluragala/building-rag-pipelines.git"

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# Core deps. `openai`+`tiktoken` enable the real stack; the rest power loaders/retrieval.
_pip("numpy", "openai", "tiktoken", "rank-bm25", "beautifulsoup4", "pypdf")

# Make `rag_pipeline` importable.
try:
    import rag_pipeline  # already on the path (local run, or repo already cloned)
except ModuleNotFoundError:
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
        if os.path.isdir("building-rag-pipelines"):
            sys.path.insert(0, "building-rag-pipelines")
        else:
            print("Clone failed. Upload the `rag_pipeline/` folder and `data/` via the "
                  "Colab file browser (left panel), then re-run this cell.")
    else:
        sys.path.insert(0, os.path.abspath(".."))  # notebooks/ -> repo root
    import rag_pipeline

def data_path(*parts):
    for base in ("data", "../data", "building-rag-pipelines/data"):
        p = os.path.join(base, *parts)
        if os.path.exists(p):
            return p
    return os.path.join("data", *parts)

print("rag_pipeline", rag_pipeline.__version__, "ready.  Colab:", IN_COLAB)

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDERS (agnostic by design)
# ============================================================
# Default stack = OpenAI (gpt-4o-mini + text-embedding-3-small).
# With no key, the package auto-falls back to the offline MOCK so the class runs.
import os
from getpass import getpass

# --- Option A: real OpenAI stack (best quality) — uncomment to enter a key ---
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (leave blank to skip): ") or ""
# In Colab you can instead use the secrets manager:
#   from google.colab import userdata; os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# --- Option B: force fully offline mock (no key, deterministic) ---
if not os.getenv("OPENAI_API_KEY"):
    os.environ["RAG_LLM_PROVIDER"] = "mock"
    os.environ["RAG_EMBED_PROVIDER"] = "mock"

from rag_pipeline import config
print(config.current_config())

## WHY — why RAG exists at all

A standalone LLM answers **only from what it memorised during training**. That
creates four hard limits:

1. **Stale knowledge** — it can't know anything after its training cutoff.
2. **No private data** — it never saw your company's docs, so it can't answer about them.
3. **Hallucination** — asked something it doesn't know, it often invents a fluent, wrong answer.
4. **No provenance** — it can't cite a source, so you can't verify or audit the answer.

> **What happens if we ignore all this and just "use a bigger LLM"?** You get a
> more eloquent hallucination. Scale doesn't add *your* facts or *fresh* facts.

**Retrieval-Augmented Generation (RAG)** fixes this by splitting the job in two:
a **retriever** fetches relevant text from a knowledge base, and a **generator**
(the LLM) answers *using that retrieved text as grounding*.

## WHAT — RAG is a modular pipeline

RAG is not one model — it is a **pipeline of stages**, each a design decision:

```
          ┌─────────┐   ┌──────────┐   ┌───────────┐   ┌──────────────┐   ┌────────────┐   ┌────────────┐
  docs →  │ LOADING │ → │ CHUNKING │ → │ RETRIEVAL │ → │ AUGMENTATION │ → │ GENERATION │ → │ EVALUATION │
          └─────────┘   └──────────┘   └───────────┘   └──────────────┘   └────────────┘   └────────────┘
           ingest &      split into      find relevant   inject context     grounded LLM     measure &
           clean+meta    embeddable       chunks (dense/  into the prompt    answer + cite    diagnose
                         units            sparse/hybrid)  (stuff/mapreduce)
```

The session's thesis: **final answer quality is the *cumulative* product of every
stage's design decisions — not just the choice of LLM.** A brilliant LLM cannot
recover from a retrieval miss; if the right chunk never reaches the prompt, no
amount of model quality helps.

## WHERE failures are born (per stage)

| Stage | A poor design here causes… |
|-------|----------------------------|
| Loading | noisy text, lost structure, no metadata → everything downstream inherits the mess |
| Chunking | diluted embeddings (too big) or fragmented ideas (too small) → retrieval misses |
| Retrieval | wrong/irrelevant chunks, or the right chunk ranked too low to survive top-k |
| Augmentation | context overflow, "lost in the middle", or unlabelled context you can't cite |
| Generation | ignores context (hallucinates) or won't say "I don't know" |
| Evaluation | you optimise the wrong stage because you never measured |

Keep this table in view — we revisit it at every stage.

> ### ✋ Predict before you run
> We're about to build a tiny end-to-end RAG over 6 short 'Acme Cloud' documents and ask it a factual question. Will a plain LLM (no retrieval) and the RAG answer differ? On which kinds of questions will the gap be largest?
>
> *Write your guess down before executing the next cell. The gap between your
> prediction and the result is where the learning happens.*

In [ ]:
# A first end-to-end RAG in ~8 lines — the whole pipeline, one call.
from rag_pipeline.loaders import load_directory
from rag_pipeline.pipeline import RAGPipeline

docs = load_directory(data_path("corpus"))          # LOADING
pipe = RAGPipeline(k=3).ingest(docs)                 # CHUNK + EMBED + STORE
result = pipe.query("How much does the Growth plan cost per month?")

print("ANSWER:\n", result["answer"], "\n")
print("SOURCES:")
for s in result["sources"]:
    print("  -", s["label"], f"(score={s['score']})")

Notice three things that define RAG:
1. The answer is **grounded** in retrieved chunks, not the model's memory.
2. It comes **with sources** you can click and verify.
3. If you re-ask something the corpus can't answer, a well-built pipeline should
   **refuse** ("I don't know") rather than invent — we'll prove this in later notebooks.

In [ ]:
# The pipeline is a glass box, not a black box. `trace` shows what each stage did.
print(result["trace"].show())

## HOW — the road map for today

Each following notebook is ONE stage, taught WHY → WHAT → HOW, with:
- **from-scratch code first** (so you see the mechanics), then
- **LangChain as a parallel mapping** (the framework wrapping the same idea).

| Notebook | Stage | Focus |
|----------|-------|-------|
| 02 | Loading | ingestion, cleaning, metadata; manual vs DocumentLoaders |
| 03 | Chunking | fixed / recursive / semantic; size vs overlap trade-offs |
| 04 | Retrieval | embeddings, dense, BM25, hybrid (RRF), filtering, reranking |
| 05 | Augmentation & Generation | stuff / map-reduce / refine; grounded prompting; citations |
| 06 | Evaluation | precision@k / recall@k / MRR, faithfulness, RAGAS, eval datasets |
| 07 | Advanced RAG | Naïve / Advanced / Modular, agentic, multi-hop, Graph RAG |
| 08 | Conclusion | full pipeline + debugging a *bad RAG output* stage by stage |

**Takeaway to hold onto:** *RAG performance is determined by cumulative design
decisions across all stages, not just the LLM.*